In [ ]:
# Project-root setup (auto-injected during notebook reorg 2026-05-06).
import sys as _sys
from pathlib import Path as _Path
_p = _Path.cwd().resolve()
while _p != _p.parent and not (_p / "pyproject.toml").exists():
    _p = _p.parent
PROJECT_ROOT = _p if (_p / "pyproject.toml").exists() else _Path.cwd()
if str(PROJECT_ROOT) not in _sys.path:
    _sys.path.insert(0, str(PROJECT_ROOT))
# Also add notebook_modules/ so `from twopoint_lockin import ...` etc. work
# without per-notebook sys.path tweaks. The five .py modules used by these
# notebooks live there now (lockin_extensions, multipoint_lockin_program,
# nv_magnetometry_analysis, odmr_sensitivity, twopoint_lockin).
_nb_modules_dir = PROJECT_ROOT / "notebook_modules"
if _nb_modules_dir.exists() and str(_nb_modules_dir) not in _sys.path:
    _sys.path.insert(0, str(_nb_modules_dir))
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

# Two-point parked lock-in module

Same hardware path as `Lockin_module.ipynb`, but tracking **two** parked frequencies on
**one** ODMR transition instead of 16 parked frequencies across 8 transitions.

| | `Lockin_module.ipynb` | this notebook |
|---|---|---|
| parked frequencies | 16 (8 transitions x 2 flanks) | 2 (1 transition x 2 flanks) |
| peak choice | all transitions that survive pair rejection | the single most pronounced + most symmetric peak |
| output | vector `dBx, dBy, dBz` reconstruction | shift of that one resonance, `df(t)` |
| dwell per batch | 16 x (sig + ref) | 2 x (sig + ref) -> ~8x faster batches |

**No magnetic-field reconstruction here.** Two points constrain one resonance frequency,
which is the field projection along that one NV axis (up to the transition's `df/dB`), not
a vector. What you get is the peak-shift time series, the same quantity produced by
`QDM Heart on Diamond/notebook/full_odmr_calibrated_peak_shift.ipynb`, but sampled at the
FPGA batch rate instead of one point per camera sweep.

Both conversion methods from the QDM-heart notebook are carried over:

* **Method A - exact Lorentzian ratio inversion.** Dip depths `d_i = B - z_i` obey
  `d_i = C g^2 / ((f_i - f0)^2 + g^2)`. The ratio `r = d_minus / d_plus` cancels the
  contrast `C` (and any common laser-power drift), leaving a quadratic in `f0` whose only
  extra input is the linewidth `g = FWHM/2`. Exact for arbitrary shifts and asymmetric
  parking.
* **Method B - linear slope estimate.** `df = [(z_plus - z_minus) - (b_plus - b_minus)] / (m_minus - m_plus)`,
  identical to `nv_toolkit.two_point.estimate_delta_f_mhz`. Valid for shifts small compared
  to the linewidth; used as a cross-check. Divergence between A and B flags that the peak
  moved a significant fraction of its linewidth.

Run order: setup -> ODMR sweep -> Step 1 (pick peak) -> Step 2 (one batch) ->
Step 3 (calibrate) -> Step 4 (live) -> Step 5 (post-process).

# Import and setup

In [ ]:
%load_ext autoreload
%matplotlib inline
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from copy import copy
import qickdawg as qd
from scipy.optimize import curve_fit

# ===== EDIT THIS: Set your RFSoC IP address =====
RFSOC_IP = '192.168.0.103'

qd.start_client(RFSOC_IP)
print(f"Connected to RFSoC at {RFSOC_IP}")

In [ ]:
default_config = qd.NVConfiguration()

# ADC channel: 0 or 1 (run diagnostic above if unsure which has your photodiode)
# 0 for ADC_D and 1 for ADC_C
default_config.adc_channel = 0
default_config.mw_channel = 1
default_config.mw_nqz = 1
default_config.mw_gain = 10500
default_config.laser_gate_pmod = 0
default_config.relax_delay_tns = 500000
default_config.readout_integration_tus = 213  # max ~106.7 us; auto-sets _treg and _tns

print("Default configuration:")
print(f"  ADC channel:     {default_config.adc_channel}")
print(f"  MW channel:      {default_config.mw_channel}")
print(f"  MW gain:         {default_config.mw_gain}")
print(f"  Relax delay:     {default_config.relax_delay_tns:.0f} ns")
print(f"  Readout window:  {default_config.readout_integration_tus:.1f} us")

In [ ]:
# ===== DSA ATTENUATION SETUP (fix 8100-count railing) =====
# Retrieve the soc proxy from qickdawg
import qickdawg as qd
soc = qd.soc  # Access the global soc proxy from start_client()

# Apply DSA to ADC_D (NV PL channel, block "00")
dsa_db = 0  # Start with 6 dB; tune as needed (0-27 dB)
soc.set_adc_attenuator("00", dsa_db)
confirmed_db = soc.get_adc_attenuator("00")
print(f"OK ADC_D DSA set to {confirmed_db} dB")
print(f"  Expected peak PL: ~{int(8100 * 10**(-confirmed_db/20))} counts (was ~8100)")

# ODMR reference sweep

The two-point tracker is only as good as the reference sweep it is calibrated against:
the linewidth, baseline and flank slopes all come from this one spectrum. Take it at the
**same bias field, MW power and laser power** you intend to run the live tracker at.

Use a fine `MW_FREQ_STEP_MHZ` (0.25-0.5 MHz) around the peak you plan to track - the
Lorentzian fit and the empirical max-slope placement in Step 1 both improve with it.

In [ ]:
# =============================================================================
# ODMR sweep - edit ALL parameters below, then run this cell once.
# Builds config, runs LockinODMR, saves full sweep to CSV (MW on / MW off / contrast).
# =============================================================================
import pandas as pd
from pathlib import Path
from datetime import datetime
import importlib

from IPython.display import FileLink, display

# --- Microwave & timing ---
ODMR_MW_GAIN = 10500              # 0-32767; up if contrast weak, down if saturated
ODMR_PRE_INIT = True             # MW+laser prepulse before sweep
ODMR_REPS = 20                   # averages per frequency point
ODMR_RELAX_DELAY_TREG = 1000    # delay register units between MW on/off segments. Normal = 500, Completely MW Off = 500000. OG = 1000

# --- ADC readout integration (photodiode PL averaging time) ---
ODMR_READOUT_INTEGRATION_TUS = 213   # or set e.g. 50.0 (microseconds). OG = 213

# --- Frequency sweep (MHz) ---
MW_FREQ_START_MHZ = 2700
MW_FREQ_STOP_MHZ = 3050
MW_FREQ_STEP_MHZ = 1             # step size (MHz); 0.5 or 0.25 gives a much better
                                 # Lorentzian fit + max-slope placement in Step 1

# --- CSV export ---
SAVE_ODMR_CSV = True             # set False to only acquire (still defines d, prog_odmr for plots)

# --- If reference should be flat but shows dips: try True (explicit low-gain pulse in ref. window) ---
ODMR_REFERENCE_ZERO_GAIN_PULSE = False
ODMR_REFERENCE_PULSE_GAIN = 1       # use 1 if gain 0 faults; increase only if needed
# --- Only if reference is flat and signal is not (mis-labeled shots), try True ---
ODMR_LOCKIN_SWAP_SIGNAL_REFERENCE = False

# --- Build configuration (inherits adc_channel, mw_channel, laser from default_config) ---
config_odmr = copy(default_config)
config_odmr.odmr_reference_zero_gain_pulse = ODMR_REFERENCE_ZERO_GAIN_PULSE
config_odmr.odmr_reference_pulse_gain = ODMR_REFERENCE_PULSE_GAIN
config_odmr.odmr_lockin_swap_signal_reference = ODMR_LOCKIN_SWAP_SIGNAL_REFERENCE
config_odmr.readout_integration_tus = ODMR_READOUT_INTEGRATION_TUS
config_odmr.mw_gain = ODMR_MW_GAIN
config_odmr.pre_init = ODMR_PRE_INIT
config_odmr.reps = ODMR_REPS
config_odmr.relax_delay_treg = ODMR_RELAX_DELAY_TREG
config_odmr.add_linear_sweep(
    "mw", "fMHz",
    start=MW_FREQ_START_MHZ,
    stop=MW_FREQ_STOP_MHZ,
    delta=MW_FREQ_STEP_MHZ,
)

print("ODMR configuration:")
print(f"  ADC channel: {config_odmr.adc_channel}  MW channel: {config_odmr.mw_channel}  gain: {config_odmr.mw_gain}")
print(f"  Sweep: {config_odmr.mw_start_fMHz:.3f} -> {config_odmr.mw_end_fMHz:.3f} MHz  ({config_odmr.nsweep_points} pts)")
print(f"  reps: {config_odmr.reps}  est. time: {qd.LockinODMR(config_odmr).total_time():.1f} s")

prog_odmr = qd.LockinODMR(config_odmr)
d = prog_odmr.acquire(progress=True)
print("Acquisition done.")

if SAVE_ODMR_CSV:
    odmr_df = pd.DataFrame(
        {
            "frequency_MHz": np.asarray(d.frequencies, dtype=float),
            "photoluminescence_mw_on_ADC": np.asarray(d.signal, dtype=float),
            "photoluminescence_mw_off_ADC": np.asarray(d.reference, dtype=float),
            "contrast_ADC": np.asarray(d.contrast, dtype=float),
            "contrast_percent": np.asarray(d.contrast_percent, dtype=float),
        }
    )
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    # Save ODMR sweep CSV to the categorized data store under PROJECT_ROOT/data/odmr_sweeps/
    # so all ODMR runs land in one place regardless of which subfolder the
    # notebook is launched from. PROJECT_ROOT is set by the project_root_setup cell.
    odmr_csv_dir = PROJECT_ROOT / "data" / "odmr_sweeps"
    odmr_csv_dir.mkdir(parents=True, exist_ok=True)
    csv_path = odmr_csv_dir / f"odmr_sweep_{ts}.csv"
    odmr_df.to_csv(csv_path, index=False)
    LAST_ODMR_CSV = csv_path
    LAST_ODMR_DF = odmr_df.copy()
    print(f"CSV: {len(odmr_df)} rows -> {csv_path.resolve()}")
    display(FileLink(csv_path.name))

In [ ]:
qd.LockinODMR.plot_sequence(config_odmr)

# Plot ODMR spectrum with subplots for MW On, MW Off, and Contrast
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

axes[0].plot(d.frequencies, d.signal, label='MW On (signal)', color='steelblue')
axes[0].set_ylabel('PL Intensity (ADC units)')
axes[0].set_title('ODMR: MW On (signal)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(d.frequencies, d.reference, label='MW Off (reference)', color='orange')
axes[1].set_ylabel('PL Intensity (ADC units)')
axes[1].set_title('ODMR: MW Off (reference)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(d.frequencies, d.contrast, label='Contrast (signal - reference)', color='green')
axes[2].set_xlabel('Frequency (MHz)')
axes[2].set_ylabel('Contrast (ADC units)')
axes[2].set_title('ODMR: Contrast')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Two-point lock-in

## Step 1a - Pick the single best peak to track

Every transition in the sweep is fitted, then scored on three things:

* **Pronounced** - dip depth divided by the local high-frequency noise sigma (SNR).
  Deeper dip -> steeper flanks -> more Hz of peak shift per ADC count.
* **Symmetric** - the left and right half-depth half-widths must match, and the two flank
  slopes must match in magnitude. Asymmetry (unresolved hyperfine, an overlapping
  neighbour, a sloping baseline) biases the two-point inversion, because both methods
  assume the same lineshape on either side of `f0`.
* **Well fitted** - RMS residual of a single-Lorentzian fit, relative to the dip depth. A
  poor fit means the single-Lorentzian model the inversion relies on is wrong here.

Four conditions are hard filters rather than scores, and a candidate failing any of them
is listed with its `reject_reason`:

* **Isolation** from the nearest other transition, in units of the peak's own linewidth
  (`MIN_ISOLATION_FWHM`) - a 14 MHz-wide line needs far more room than a 2 MHz one. If a
  neighbour sits inside the fit window, the flank the tracker parks on is not the flank
  the calibration measured.
* **Resolution** - at least `MIN_POINTS_PER_FWHM` sweep points across the linewidth. Every
  kHz this notebook reports is scaled by that linewidth, and a linewidth cannot be measured
  from a sweep that barely samples it.
* **The fit converged** - `gamma` must not sit on either bound.
* **`f0` is actually inside a dip** - if the half-depth crossings are undefined, the
  detector collapsed a hyperfine pair and fitted the bump between its components.

If nothing survives, the cell **raises** with the per-candidate reasons and the sweep step
you would need instead, rather than quietly calibrating on a bad peak. Setting
`FORCE_PEAK_CENTER_MHZ` overrides that and proceeds anyway.

The Lorentzian fit runs **twice**: once in the detection window `FIT_WINDOW_MHZ`, then
again in a window scaled to the linewidth that first pass measured
(`PEAK_FIT_WINDOW_FACTOR x FWHM`, clipped so it cannot swallow a neighbour). Without the
second pass a line much broader than `FIT_WINDOW_MHZ` is fitted over a nearly straight
slice of its own flank, and both the FWHM and the parked placement come out wrong.

Set `FORCE_PEAK_CENTER_MHZ` to override the automatic choice. Step 1b then places the
parked pair on whichever peak this cell selected.

In [ ]:
# Step 1a - Choose ONE peak to track.
#
# Difference from Lockin_module: that notebook parks 16 frequencies (8 transitions x 2
# flanks) so nv_toolkit can reconstruct a full B vector. Here we park TWO frequencies on
# ONE transition. That tracks the shift of that resonance -- the field projection along
# that NV axis -- but is NOT enough for a vector reconstruction, so no reconstruction
# cell exists in this notebook.
#
# Candidate detection reuses the same nv_toolkit pipeline as Lockin_module
# (detect -> cluster -> collapse -> local fit -> pair rejection); only the selection,
# the linewidth-scaled refit and the placement that follow are new.
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from IPython.display import FileLink, display

# ------------------------------------------------------------------ knobs ---
ODMR_CSV_OVERRIDE = None        # Path("...odmr_sweep_....csv") to analyse a saved sweep

# Candidate detection (same values as Lockin_module).
SNR_THRESHOLD         = 0.8
CLUSTER_MERGE_MHZ     = 3.0
FIT_WINDOW_MHZ        = 8.0     # detection-stage window; the refit below rescales it
MAX_CENTERS           = 8
REQUIRE_PAIRED_PEAKS  = True    # drop fitted dips with no 2870-mirrored partner
PEAK_SEARCH_RANGE_MHZ = None    # e.g. (2820, 2920) to restrict where we look

# Second-pass fit and analysis windows, in units of the measured FWHM.
PEAK_FIT_WINDOW_FACTOR = 3.0    # full refit window = factor x FWHM
SLOPE_SEARCH_FWHM      = 1.5    # flank max-slope search half-width = this x FWHM

# Peak scoring (weights need not sum to 1; they are renormalised).
W_PROMINENCE       = 0.45
W_SYMMETRY         = 0.40
W_FIT              = 0.15
MIN_SNR            = 3.0        # reject candidates shallower than this
MIN_ISOLATION_MHZ  = 5.0        # absolute floor on the distance to the nearest neighbour
MIN_ISOLATION_FWHM = 1.5        # ... and at least this many of the peak's own linewidths
MIN_POINTS_PER_FWHM = 4.0       # the sweep must resolve the line this well to calibrate it
FIT_RESID_SCALE    = 0.10       # residual/depth that halves the fit sub-score

# Manual override: a frequency in MHz forces which peak is tracked (nearest candidate).
FORCE_PEAK_CENTER_MHZ = None

# ------------------------------------------------------- nv_toolkit import ---
_search_roots = [Path.cwd(), *Path.cwd().parents]
NV_REPO_DIR = None
for _root in _search_roots:
    candidate = _root / "NV_Magnetometry_Software-main"
    if (candidate / "nv_toolkit" / "operator_cli.py").exists():
        NV_REPO_DIR = candidate
        break
if NV_REPO_DIR is None:
    raise FileNotFoundError("NV_Magnetometry_Software-main not found relative to notebook directory")
if str(NV_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(NV_REPO_DIR))

from nv_toolkit.peaks import (
    detect_baseline_corrected_dip_candidates,
    cluster_dip_candidates,
    collapse_candidate_clusters,
    fit_local_dips,
    reject_unpaired_peaks,
)
from nv_toolkit.tui import _suggest_parked_frequencies

# ----------------------------------------------------------- load the sweep ---
if ODMR_CSV_OVERRIDE is not None:
    ODMR_CSV_FOR_PLAN = Path(ODMR_CSV_OVERRIDE)
elif "LAST_ODMR_CSV" in globals() and Path(LAST_ODMR_CSV).exists():
    ODMR_CSV_FOR_PLAN = Path(LAST_ODMR_CSV)
else:
    _odmr_dir = PROJECT_ROOT / "data" / "odmr_sweeps"
    _odmr_files = sorted(_odmr_dir.glob("odmr_sweep_*.csv"))
    if not _odmr_files:
        raise FileNotFoundError(f"No odmr_sweep_*.csv found in {_odmr_dir}")
    ODMR_CSV_FOR_PLAN = _odmr_files[-1]
print(f"Reference ODMR: {ODMR_CSV_FOR_PLAN}")

_odmr_df = pd.read_csv(ODMR_CSV_FOR_PLAN)
freqs_mhz = _odmr_df["frequency_MHz"].to_numpy(dtype=float)
_mw_on  = _odmr_df["photoluminescence_mw_on_ADC"].to_numpy(dtype=float)
_mw_off = _odmr_df["photoluminescence_mw_off_ADC"].to_numpy(dtype=float)
_df_step_mhz = float(np.median(np.diff(freqs_mhz)))

# TWO normalisations, deliberately kept apart (same split as Lockin_module):
#   `measured`      = mw_on / |median(mw_on)|  -> dip DETECTION and fitting only.
#   `spectrum_norm` = mw_on / |median(mw_off)| -> CALIBRATION scale. It matches the live
#     per-point mw_on/|mw_off| ratio, so baselines and slopes live on the same scale as
#     the parked measurements. MW-off has no dips, so its MEDIAN is used rather than the
#     per-bin value, which would inject reference noise into the calibration slopes.
measured      = _mw_on / abs(float(np.nanmedian(_mw_on)))
spectrum_norm = _mw_on / abs(float(np.median(_mw_off)))

# ------------------------------------------------- candidate detection chain ---
candidates = detect_baseline_corrected_dip_candidates(
    freqs_mhz, measured, min_prominence_sigma=SNR_THRESHOLD,
)
clusters = cluster_dip_candidates(candidates, merge_distance_mhz=CLUSTER_MERGE_MHZ)
candidate_centers, _cluster_rows = collapse_candidate_clusters(clusters, max_centers=MAX_CENTERS)

if PEAK_SEARCH_RANGE_MHZ is not None:
    _lo, _hi = float(PEAK_SEARCH_RANGE_MHZ[0]), float(PEAK_SEARCH_RANGE_MHZ[1])
    _n_before = len(candidate_centers)
    candidate_centers = candidate_centers[(candidate_centers >= _lo) & (candidate_centers <= _hi)]
    print(f"Search restricted to {_lo:.0f}-{_hi:.0f} MHz: kept {len(candidate_centers)} of {_n_before} centers.")
    if len(candidate_centers) == 0:
        raise RuntimeError("No candidate centers remain after PEAK_SEARCH_RANGE_MHZ filtering.")

local_fits = fit_local_dips(
    freqs_mhz, measured, candidate_centers,
    window_mhz=FIT_WINDOW_MHZ, hyperfine_merge_mhz=0.0,
)
if REQUIRE_PAIRED_PEAKS:
    kept_fits, rejected_fits = reject_unpaired_peaks(
        local_fits, (float(np.nanmin(freqs_mhz)), float(np.nanmax(freqs_mhz))),
    )
    if not kept_fits:
        print("Pair rejection removed every peak; falling back to all fitted peaks "
              "(set REQUIRE_PAIRED_PEAKS = False to silence this).")
        kept_fits, rejected_fits = list(local_fits), []
else:
    kept_fits, rejected_fits = list(local_fits), []
kept_fits = sorted(kept_fits, key=lambda f: f.center_mhz)
if not kept_fits:
    raise RuntimeError("No fitted transitions to choose from.")

# All detected centers (kept + rejected) define isolation distances -- a rejected peak
# still contaminates a flank parked next to it.
_all_centers = np.sort(np.asarray(
    [f.center_mhz for f in kept_fits] + [f.center_mhz for f in rejected_fits], dtype=float
))

# --------------------------------------------------------------- lineshape ---
def lorentzian_dip(f, baseline, contrast, f0, gamma):
    """Single Lorentzian dip in the calibration (z) scale."""
    return baseline - contrast * gamma**2 / ((f - f0)**2 + gamma**2)


def noise_sigma(y):
    """Robust noise sigma from the MAD of SECOND differences.

    Second differences annihilate any linear ramp, so the estimate is not inflated by
    the broad dip flanks the way a first-difference estimate would be. For white noise
    var(d2) = 6 sigma^2, hence the sqrt(6).
    """
    d2 = np.diff(np.asarray(y, dtype=float), n=2)
    if d2.size < 3:
        return float("nan")
    mad = np.median(np.abs(d2 - np.median(d2)))
    return float(1.4826 * mad / np.sqrt(6.0))


def orientation(freqs, y, center, half_width):
    """+1 if the feature at `center` points DOWN in y, -1 if it points up.

    Homodyne ADC can come back negative, in which case dips point up in the raw
    normalised trace. Everything downstream works in z = orient * y, where the dip
    always points down and the Lorentzian model above applies verbatim.
    """
    m = np.abs(freqs - center) <= half_width
    if m.sum() < 5:
        return 1.0
    y_center = float(np.interp(center, freqs, y))
    return 1.0 if y_center < float(np.median(y[m])) else -1.0


def fit_peak_z(freqs, y, center_guess, window_mhz, orient=None):
    """Orientation-aware single-Lorentzian fit in the calibration scale.

    Returns a dict in z-space: baseline, contrast, f0, gamma, fwhm, rms_resid, orient.
    """
    half = 0.5 * float(window_mhz)
    if orient is None:
        orient = orientation(freqs, y, center_guess, half)
    m = np.abs(freqs - center_guess) <= half
    fw, zw = freqs[m], orient * y[m]
    if fw.size < 6:
        raise RuntimeError(f"Only {fw.size} points within +/-{half:.1f} MHz of {center_guess:.2f} MHz")

    span = float(fw.max() - fw.min())
    dfreq = float(np.median(np.diff(fw)))
    b0 = float(np.median(zw[zw >= np.percentile(zw, 60)]))
    depth0 = max(b0 - float(np.min(zw)), 1e-9)
    f0_0 = float(fw[np.argmin(zw)])
    _below = fw[zw < b0 - depth0 / 2]
    gamma0 = max(0.5 * float(_below.max() - _below.min()), dfreq) if _below.size >= 2 else 3 * dfreq
    z_range = max(float(zw.max() - zw.min()), 1e-9)

    p0 = [b0, depth0, f0_0, gamma0]
    bounds = (
        [b0 - 5 * z_range, 0.0,          fw.min(), dfreq / 4],
        [b0 + 5 * z_range, 20 * z_range, fw.max(), span],
    )
    popt, _ = curve_fit(lorentzian_dip, fw, zw, p0=p0, bounds=bounds, maxfev=20000)
    resid = zw - lorentzian_dip(fw, *popt)
    # A gamma sitting on either bound means the fit never converged on a real linewidth --
    # usually the sweep step is too coarse for the line, or the window caught no curvature.
    _g_lo, _g_hi = bounds[0][3], bounds[1][3]
    railed = bool(popt[3] <= _g_lo * 1.01 or popt[3] >= _g_hi * 0.99)
    return {
        "gamma_railed": railed,
        "orient":     float(orient),
        "baseline":   float(popt[0]),
        "contrast":   float(popt[1]),
        "f0_mhz":     float(popt[2]),
        "gamma_mhz":  float(popt[3]),
        "fwhm_mhz":   float(2 * popt[3]),
        "rms_resid":  float(np.sqrt(np.mean(resid**2))),
        "n_points":   int(fw.size),
        "window_mhz": float(window_mhz),
    }


def fit_peak_two_pass(freqs, y, center_guess, isolation_mhz):
    """Fit once in FIT_WINDOW_MHZ, then refit in a window scaled to the measured FWHM.

    A line much broader than FIT_WINDOW_MHZ is otherwise fitted over a nearly straight
    slice of its own flank, which lets (contrast, gamma) run off together. The refit
    window is clipped so it cannot reach the nearest neighbouring transition.
    """
    first = fit_peak_z(freqs, y, center_guess, FIT_WINDOW_MHZ)
    want = PEAK_FIT_WINDOW_FACTOR * first["fwhm_mhz"]
    cap = 1.8 * float(isolation_mhz) if np.isfinite(isolation_mhz) else np.inf
    window = float(np.clip(want, FIT_WINDOW_MHZ, cap))
    if abs(window - first["window_mhz"]) < 0.5 * _df_step_mhz:
        return first
    try:
        return fit_peak_z(freqs, y, first["f0_mhz"], window, orient=first["orient"])
    except Exception:
        return first


def half_depth_halfwidths(freqs, z, f0, baseline, depth):
    """Distance from f0 out to the half-depth crossing on each side (interpolated)."""
    target = baseline - 0.5 * depth
    i0 = int(np.argmin(np.abs(freqs - f0)))
    if not (z[i0] < target):        # f0 is not actually inside the dip
        return float("nan"), float("nan")

    def walk(step):
        i = i0
        for _ in range(freqs.size):
            j = i + step
            if j < 0 or j >= freqs.size:
                return float("nan")
            if z[j] >= target:
                if z[j] == z[i]:
                    return abs(float(freqs[j]) - f0)
                t = (target - z[i]) / (z[j] - z[i])
                return abs(float(freqs[i] + t * (freqs[j] - freqs[i])) - f0)
            i = j
        return float("nan")

    return walk(-1), walk(+1)


def flank_max_slopes(freqs, z, f0, half_width):
    """Empirical steepest-slope point and slope on each flank, within +/- half_width."""
    grad = np.gradient(z, freqs)
    out = {}
    for side, mask in (
        ("minus", (freqs < f0) & (freqs >= f0 - half_width)),
        ("plus",  (freqs > f0) & (freqs <= f0 + half_width)),
    ):
        if mask.sum() < 2:
            out[side] = (float("nan"), float("nan"))
            continue
        idx = np.flatnonzero(mask)[int(np.argmax(np.abs(grad[mask])))]
        out[side] = (float(freqs[idx]), float(grad[idx]))
    return out

# ------------------------------------------------------------ score candidates ---
_sigma_global = noise_sigma(spectrum_norm)
score_rows = []
fit_cache = {}
for fit in kept_fits:
    center = float(fit.center_mhz)
    others = _all_centers[np.abs(_all_centers - center) > 1e-6]
    isolation = float(np.min(np.abs(others - center))) if others.size else float("inf")

    try:
        pk = fit_peak_two_pass(freqs_mhz, spectrum_norm, center, isolation)
    except Exception as exc:
        print(f"  {center:8.2f} MHz: Lorentzian fit failed ({exc}) - skipped")
        continue
    fit_cache[center] = pk

    z = pk["orient"] * spectrum_norm
    depth = pk["contrast"]
    snr = depth / _sigma_global if _sigma_global > 0 else float("nan")

    w_left, w_right = half_depth_halfwidths(freqs_mhz, z, pk["f0_mhz"], pk["baseline"], depth)
    if np.isfinite(w_left) and np.isfinite(w_right) and (w_left + w_right) > 0:
        asym_width = abs(w_left - w_right) / (w_left + w_right)
    else:
        asym_width = 1.0

    slope_half = max(SLOPE_SEARCH_FWHM * pk["fwhm_mhz"], 3 * _df_step_mhz)
    slopes = flank_max_slopes(freqs_mhz, z, pk["f0_mhz"], slope_half)
    m_l, m_r = abs(slopes["minus"][1]), abs(slopes["plus"][1])
    asym_slope = abs(m_l - m_r) / (m_l + m_r) if np.isfinite(m_l + m_r) and (m_l + m_r) > 0 else 1.0

    sym_score = 1.0 - 0.5 * (min(asym_width, 1.0) + min(asym_slope, 1.0))
    fit_score = 1.0 / (1.0 + (pk["rms_resid"] / depth) / FIT_RESID_SCALE) if depth > 0 else 0.0

    points_per_fwhm = pk["fwhm_mhz"] / _df_step_mhz if _df_step_mhz > 0 else np.inf
    reasons = []
    if not (np.isfinite(w_left) and np.isfinite(w_right)):
        reasons.append("f0 not inside a dip (fit landed between components?)")
    if pk["gamma_railed"]:
        reasons.append("linewidth railed on a fit bound")
    if points_per_fwhm < MIN_POINTS_PER_FWHM:
        reasons.append(f"only {points_per_fwhm:.1f} sweep points per FWHM")
    if snr < MIN_SNR:
        reasons.append(f"SNR {snr:.1f} < {MIN_SNR}")

    score_rows.append({
        "center_mhz":     pk["f0_mhz"],
        "points_per_fwhm": points_per_fwhm,
        "reject_reason":  "; ".join(reasons),
        "fwhm_mhz":       pk["fwhm_mhz"],
        "fit_window_mhz": pk["window_mhz"],
        "depth":          depth,
        "snr":            snr,
        "isolation_mhz":  isolation,
        "halfwidth_left_mhz":  w_left,
        "halfwidth_right_mhz": w_right,
        "asym_width":     asym_width,
        "asym_slope":     asym_slope,
        "rms_resid_rel":  pk["rms_resid"] / depth if depth > 0 else np.inf,
        "sym_score":      sym_score,
        "fit_score":      fit_score,
        "detect_center_mhz": center,
    })

if not score_rows:
    raise RuntimeError("Every candidate failed the Lorentzian fit -- check the sweep range/step.")

df_scores = pd.DataFrame(score_rows)
_snr_max = float(df_scores["snr"].max())
df_scores["prom_score"] = df_scores["snr"] / _snr_max if _snr_max > 0 else 0.0
_wsum = W_PROMINENCE + W_SYMMETRY + W_FIT
df_scores["total_score"] = (
    W_PROMINENCE * df_scores["prom_score"]
    + W_SYMMETRY * df_scores["sym_score"]
    + W_FIT * df_scores["fit_score"]
) / _wsum
df_scores["isolation_needed_mhz"] = np.maximum(
    MIN_ISOLATION_MHZ, MIN_ISOLATION_FWHM * df_scores["fwhm_mhz"])
df_scores["eligible"] = (
    (df_scores["reject_reason"] == "")
    & (df_scores["isolation_mhz"] >= df_scores["isolation_needed_mhz"])
)
df_scores.loc[
    (df_scores["reject_reason"] == "")
    & (df_scores["isolation_mhz"] < df_scores["isolation_needed_mhz"]),
    "reject_reason",
] = "neighbour too close"
df_scores = df_scores.sort_values("total_score", ascending=False).reset_index(drop=True)

_eligible = df_scores[df_scores["eligible"]]
if _eligible.empty:
    _narrowest = float(df_scores["fwhm_mhz"].min())
    _msg = [
        "No candidate is usable for a two-point calibration. Per-candidate reasons:",
        *(f"    {r.center_mhz:9.3f} MHz : {r.reject_reason}" for r in df_scores.itertuples()),
        "",
        f"Sweep step is {_df_step_mhz:.3f} MHz and the narrowest fitted linewidth is "
        f"{_narrowest:.3f} MHz.",
    ]
    if _narrowest / _df_step_mhz < MIN_POINTS_PER_FWHM:
        _msg.append(
            f"    -> Re-run the ODMR sweep with MW_FREQ_STEP_MHZ <= "
            f"{_narrowest / MIN_POINTS_PER_FWHM:.2f} MHz (narrow the range to keep it quick). "
            f"A linewidth cannot be calibrated from a sweep that barely samples it, and every "
            f"kHz this notebook reports is scaled by that linewidth."
        )
    _msg.append("    -> Or set FORCE_PEAK_CENTER_MHZ to proceed anyway, accepting the above.")
    if FORCE_PEAK_CENTER_MHZ is None:
        display(df_scores[["center_mhz", "fwhm_mhz", "points_per_fwhm", "snr",
                           "isolation_mhz", "total_score", "reject_reason"]].round(4))
        raise RuntimeError("\n".join(_msg))
    print("\n".join(_msg))
    _eligible = df_scores

if FORCE_PEAK_CENTER_MHZ is not None:
    _i = int(np.argmin(np.abs(df_scores["center_mhz"].to_numpy() - float(FORCE_PEAK_CENTER_MHZ))))
    chosen = df_scores.iloc[_i]
    print(f"FORCE_PEAK_CENTER_MHZ = {FORCE_PEAK_CENTER_MHZ} -> using the candidate at "
          f"{chosen['center_mhz']:.3f} MHz (rank {_i + 1} of {len(df_scores)}).")
else:
    chosen = _eligible.iloc[0]

CHOSEN_PEAK = fit_cache[float(chosen["detect_center_mhz"])]
Z_ORIENT = CHOSEN_PEAK["orient"]
z_ref = Z_ORIENT * spectrum_norm

print("\nCandidate ranking (higher total_score is better):")
display(df_scores[[
    "center_mhz", "fwhm_mhz", "points_per_fwhm", "snr", "isolation_mhz", "isolation_needed_mhz",
    "asym_width", "asym_slope", "rms_resid_rel", "prom_score", "sym_score", "fit_score",
    "total_score", "eligible", "reject_reason",
]].round(4))

## Step 1b - Place the parked pair and build the calibration

Split from Step 1a so placement can be re-tuned - a different `PLACEMENT_MODE`, a manual
pair, a different peak via `FORCE_PEAK_CENTER_MHZ` - without re-running detection and
scoring on the whole sweep.

Placement puts the pair at the empirical steepest-slope point on each flank, symmetrised,
then (`BALANCE_PARKED_PAIR`) slides the pair as a unit onto the **balance point** - the
centre at which both flanks read the same normalised PL. That is the true zero of the
two-point discriminator. On a real, slightly asymmetric line it sits a few tens of kHz off
the fitted Lorentzian centre, and parking symmetrically about the fit instead leaves a
constant offset in the error signal.

The cell ends by building `TWOPOINT_CALIB` - `f0`, `FWHM`, contrast, baseline, the two
parked frequencies and their measured baselines/slopes - plus the three conversion helpers
(documented in Step 3) that every later cell consumes.

In [ ]:
# Step 1b - Place the two parked frequencies and build TWOPOINT_CALIB.
# Consumes CHOSEN_PEAK / z_ref / freqs_mhz from Step 1a; re-runnable on its own after
# changing any knob below.
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import FileLink, display

if "CHOSEN_PEAK" not in globals():
    raise RuntimeError("Run Step 1a first -- it selects the peak this cell calibrates.")

# Parked-frequency placement on the chosen peak:
#   "symmetric_max_slope"  - empirical max |dz/df| per flank, then symmetrised (default)
#   "symmetric_half_max"   - f0 +/- FWHM/2
#   "lorentzian_max_slope" - f0 +/- gamma/sqrt(3) (analytic Lorentzian max-slope point)
#   "toolkit"              - nv_toolkit _suggest_parked_frequencies (Lockin_module rule)
#   "manual"               - use PARKED_MANUAL_MHZ
PLACEMENT_MODE    = "symmetric_max_slope"
PARKED_MANUAL_MHZ = None        # (f_minus, f_plus) when PLACEMENT_MODE == "manual"
# Slide the pair (keeping its spacing) onto the measured balance point, where the two
# flanks read the same z. That is the true zero of the two-point discriminator; the fitted
# Lorentzian center is generally a few tens of kHz away from it on real, slightly
# asymmetric lines. Ignored for PLACEMENT_MODE == "manual".
BALANCE_PARKED_PAIR = True

# NV gyromagnetic ratio, used only to quote a peak shift as an equivalent field
# projection along the tracked NV axis. This is NOT |B|.
GAMMA_NV_MHZ_PER_NT = 28.024e-6

# -------------------------------------------------- place the two parked freqs ---
f0 = CHOSEN_PEAK["f0_mhz"]
gamma = CHOSEN_PEAK["gamma_mhz"]
PEAK_WINDOW_HALF_MHZ = max(SLOPE_SEARCH_FWHM * CHOSEN_PEAK["fwhm_mhz"], 3 * _df_step_mhz)
_slopes = flank_max_slopes(freqs_mhz, z_ref, f0, PEAK_WINDOW_HALF_MHZ)

if PLACEMENT_MODE == "manual":
    if PARKED_MANUAL_MHZ is None:
        raise ValueError('PLACEMENT_MODE == "manual" requires PARKED_MANUAL_MHZ = (f_minus, f_plus)')
    f_minus, f_plus = sorted(float(v) for v in PARKED_MANUAL_MHZ)
elif PLACEMENT_MODE == "symmetric_half_max":
    f_minus, f_plus = f0 - gamma, f0 + gamma
elif PLACEMENT_MODE == "lorentzian_max_slope":
    _delta = gamma / np.sqrt(3.0)
    f_minus, f_plus = f0 - _delta, f0 + _delta
elif PLACEMENT_MODE == "symmetric_max_slope":
    _d_minus = abs(f0 - _slopes["minus"][0])
    _d_plus = abs(_slopes["plus"][0] - f0)
    _delta = float(np.nanmean([_d_minus, _d_plus]))
    if not np.isfinite(_delta) or _delta <= 0:
        _delta = gamma / np.sqrt(3.0)
        print("  Empirical max-slope search failed; fell back to gamma/sqrt(3).")
    f_minus, f_plus = f0 - _delta, f0 + _delta
elif PLACEMENT_MODE == "toolkit":
    _plan = _suggest_parked_frequencies(freqs_mhz, measured, np.asarray([f0]), placement="auto")
    f_minus = float(_plan[0].frequency_minus_mhz)
    f_plus = float(_plan[0].frequency_plus_mhz)
else:
    raise ValueError(f"Unknown PLACEMENT_MODE: {PLACEMENT_MODE!r}")

# Slide the pair onto the measured balance point (z equal on both flanks), keeping the
# spacing. Solved by bisection on the sign change of the imbalance, searched within half
# the parked spacing of f0 so it cannot wander onto a neighbouring feature.
BALANCE_OFFSET_MHZ = 0.0
if BALANCE_PARKED_PAIR and PLACEMENT_MODE != "manual":
    _delta = 0.5 * (f_plus - f_minus)
    _cs = np.linspace(f0 - _delta, f0 + _delta, 4001)
    _imb = (np.interp(_cs - _delta, freqs_mhz, z_ref)
            - np.interp(_cs + _delta, freqs_mhz, z_ref))
    _sc = np.flatnonzero(np.sign(_imb[:-1]) != np.sign(_imb[1:]))
    if _sc.size:
        _i = int(_sc[np.argmin(np.abs(_cs[_sc] - f0))])
        _t = _imb[_i] / (_imb[_i] - _imb[_i + 1])
        _c_bal = float(_cs[_i] + _t * (_cs[_i + 1] - _cs[_i]))
        BALANCE_OFFSET_MHZ = _c_bal - f0
        f_minus, f_plus = _c_bal - _delta, _c_bal + _delta
    else:
        print("  Balance point not bracketed within +/- one parked spacing of f0; "
              "leaving the pair centred on the fitted f0.")

# Measured baselines and slopes at the parked points, in the calibration (z) scale.
_grad_z = np.gradient(z_ref, freqs_mhz)
baseline_minus = float(np.interp(f_minus, freqs_mhz, z_ref))
baseline_plus  = float(np.interp(f_plus,  freqs_mhz, z_ref))
slope_minus    = float(np.interp(f_minus, freqs_mhz, _grad_z))
slope_plus     = float(np.interp(f_plus,  freqs_mhz, _grad_z))

TWOPOINT_CALIB = {
    "odmr_csv":              str(ODMR_CSV_FOR_PLAN),
    "orient":                float(Z_ORIENT),
    "f0_mhz":                float(f0),
    "gamma_mhz":             float(gamma),
    "fwhm_mhz":              float(CHOSEN_PEAK["fwhm_mhz"]),
    "contrast":              float(CHOSEN_PEAK["contrast"]),
    "baseline_fit":          float(CHOSEN_PEAK["baseline"]),
    "rms_resid":             float(CHOSEN_PEAK["rms_resid"]),
    "fit_window_mhz":        float(CHOSEN_PEAK["window_mhz"]),
    "f_minus_mhz":           float(f_minus),
    "f_plus_mhz":            float(f_plus),
    "baseline_minus":        baseline_minus,
    "baseline_plus":         baseline_plus,
    "slope_minus_per_mhz":   slope_minus,
    "slope_plus_per_mhz":    slope_plus,
    "placement_mode":        PLACEMENT_MODE,
    "balance_offset_mhz":    float(BALANCE_OFFSET_MHZ),
    "snr":                   float(chosen["snr"]),
    "total_score":           float(chosen["total_score"]),
    "gamma_nv_mhz_per_nT":   float(GAMMA_NV_MHZ_PER_NT),
    "f0_ref_exact_mhz":      float("nan"),   # filled in below
}

TWOPOINT_FREQS_MHZ = np.asarray([f_minus, f_plus], dtype=float)

# The parked frequencies must straddle f0 with slopes of opposite sign, or neither
# conversion method has a stable inverse.
assert f_minus < f0 < f_plus, (
    f"Parked frequencies must straddle f0: {f_minus:.3f} < {f0:.3f} < {f_plus:.3f} failed"
)
if not (slope_minus < 0 < slope_plus):
    print(f"WARNING: expected slope_minus < 0 < slope_plus in z-space, got "
          f"{slope_minus:+.4g} and {slope_plus:+.4g}. The parked points may be outside the "
          f"dip, or the peak is contaminated by a neighbour.")

# ------------------------------------------------------- conversion helpers ---
# Defined here because they are pure functions of TWOPOINT_CALIB; the maths behind them
# is documented in the Step 3 markdown. Step 3 onwards just calls them.

def twopoint_z_from_counts(signal_counts, reference_counts, calib=None):
    """Raw ADC counts -> calibration-scale z.

    Dividing by |reference| (not reference) keeps dips pointing the same way for positive
    PL and negative homodyne ADC; `orient` then flips the whole trace so the dip points
    down. Identical convention to nv_toolkit.two_point.normalised_signal_from_counts,
    with the extra orientation factor.
    """
    calib = TWOPOINT_CALIB if calib is None else calib
    ref = np.abs(np.asarray(reference_counts, dtype=float))
    if np.any(ref == 0):
        raise ValueError("reference_counts must be non-zero")
    return float(calib["orient"]) * np.asarray(signal_counts, dtype=float) / ref


def twopoint_f0_exact(z_minus, z_plus, calib=None):
    """Method A -- exact Lorentzian dip center from the two parked z values.

    Vectorised. Returns NaN where either dip depth is non-positive (a parked point sat at
    or above the fitted baseline, so the depth ratio has no physical root).
    """
    calib = TWOPOINT_CALIB if calib is None else calib
    g = float(calib["gamma_mhz"])
    f1 = float(calib["f_minus_mhz"])
    f2 = float(calib["f_plus_mhz"])
    baseline = float(calib["baseline_fit"])

    d1 = baseline - np.asarray(z_minus, dtype=float)
    d2 = baseline - np.asarray(z_plus, dtype=float)
    ok = (d1 > 0) & (d2 > 0)

    with np.errstate(invalid="ignore", divide="ignore"):
        r = np.where(ok, d1 / np.where(d2 != 0, d2, np.nan), np.nan)
        a = r - 1.0
        b = -2.0 * (r * f1 - f2)
        c = r * f1**2 - f2**2 + (r - 1.0) * g**2
        disc = b**2 - 4.0 * a * c
        sq = np.sqrt(np.where(disc >= 0, disc, np.nan))
        linear_root = -c / b                       # symmetric limit r -> 1
        quadratic = np.abs(a) > 1e-12
        root_plus = np.where(quadratic, (-b + sq) / (2.0 * a), linear_root)
        root_minus = np.where(quadratic, (-b - sq) / (2.0 * a), linear_root)

    mid = 0.5 * (f1 + f2)
    pick_plus = np.abs(root_plus - mid) <= np.abs(root_minus - mid)
    f0_out = np.where(pick_plus, root_plus, root_minus)
    return np.where(ok, f0_out, np.nan)


def twopoint_delta_f_linear(z_minus, z_plus, calib=None):
    """Method B -- slope-based df, same formula as nv_toolkit estimate_delta_f_mhz."""
    calib = TWOPOINT_CALIB if calib is None else calib
    denom = float(calib["slope_minus_per_mhz"]) - float(calib["slope_plus_per_mhz"])
    if abs(denom) < 1e-12:
        raise ValueError("Calibration slopes are too small for a stable delta_f estimate")
    d_current = np.asarray(z_plus, dtype=float) - np.asarray(z_minus, dtype=float)
    delta_d = d_current - (float(calib["baseline_plus"]) - float(calib["baseline_minus"]))
    return delta_d / denom


# Zero point for method A. Method B is zeroed by construction on the MEASURED baselines
# (b_plus - b_minus), so method A must be zeroed the same way: feed the measured baselines
# through the exact inversion and use the f0 it returns as the reference. Otherwise the
# two methods are offset by the fit-vs-data mismatch at the parked points and their
# disagreement stops being a useful diagnostic.
_f0_ref_exact = float(twopoint_f0_exact(baseline_minus, baseline_plus, TWOPOINT_CALIB))
if not np.isfinite(_f0_ref_exact):
    print("WARNING: the exact inversion has no root at the calibration baselines "
          "(a parked point sits at/above the fitted baseline). Falling back to the fitted "
          "f0 as the zero point; methods A and B will carry a constant offset.")
    _f0_ref_exact = float(f0)
TWOPOINT_CALIB["f0_ref_exact_mhz"] = _f0_ref_exact

# --------------------------------------------------------------- persist -----
TWOPOINT_DIR = PROJECT_ROOT / "data" / "twopoint_lockin"
TWOPOINT_DIR.mkdir(parents=True, exist_ok=True)
TWOPOINT_CALIB_JSON = TWOPOINT_DIR / f"twopoint_calibration_{ODMR_CSV_FOR_PLAN.stem}.json"
TWOPOINT_CALIB_JSON.write_text(json.dumps(TWOPOINT_CALIB, indent=2), encoding="utf-8")

TWOPOINT_PLAN_CSV = TWOPOINT_DIR / f"twopoint_plan_{ODMR_CSV_FOR_PLAN.stem}.csv"
pd.DataFrame([
    {"point_index": 1, "point_in_block": 0, "label": "f_minus", "frequency_mhz": f_minus,
     "baseline_z": baseline_minus, "slope_z_per_mhz": slope_minus},
    {"point_index": 2, "point_in_block": 1, "label": "f_plus", "frequency_mhz": f_plus,
     "baseline_z": baseline_plus, "slope_z_per_mhz": slope_plus},
]).to_csv(TWOPOINT_PLAN_CSV, index=False)

_denom = slope_minus - slope_plus
print(f"\nTracked peak:  f0 = {f0:.4f} MHz,  FWHM = {CHOSEN_PEAK['fwhm_mhz']:.4f} MHz,  "
      f"contrast = {CHOSEN_PEAK['contrast']:.5f},  SNR = {chosen['snr']:.1f}")
print(f"Fit:           window {CHOSEN_PEAK['window_mhz']:.1f} MHz "
      f"({CHOSEN_PEAK['n_points']} points), residual {CHOSEN_PEAK['rms_resid']:.3e} "
      f"({100 * CHOSEN_PEAK['rms_resid'] / CHOSEN_PEAK['contrast']:.1f}% of depth)")
print(f"Symmetry:      half-widths L/R = {chosen['halfwidth_left_mhz']:.3f} / "
      f"{chosen['halfwidth_right_mhz']:.3f} MHz  (asym {chosen['asym_width']:.3f}), "
      f"slope asym {chosen['asym_slope']:.3f}")
print(f"Placement:     {PLACEMENT_MODE}  ->  f- = {f_minus:.4f} MHz, f+ = {f_plus:.4f} MHz "
      f"(spacing +/-{0.5 * (f_plus - f_minus):.4f} MHz = {0.5*(f_plus-f_minus)/CHOSEN_PEAK['fwhm_mhz']:.2f} FWHM)")
if BALANCE_PARKED_PAIR and PLACEMENT_MODE != "manual":
    print(f"Balance:       pair centred {BALANCE_OFFSET_MHZ*1e3:+.1f} kHz from the fitted f0, "
          f"where both flanks read the same z (residual imbalance "
          f"{baseline_plus - baseline_minus:+.2e})")
print(f"Flank slopes:  m- = {slope_minus:+.5g} /MHz,  m+ = {slope_plus:+.5g} /MHz  "
      f"(denominator m- - m+ = {_denom:+.5g})")
print(f"Linear-method scale: 1e-4 in (z+ - z-) -> {abs(1e-4 / _denom) * 1e3:.2f} kHz of peak shift")
print(f"Zero point:    method-A reference f0 = {_f0_ref_exact:.4f} MHz "
      f"({(_f0_ref_exact - f0) * 1e3:+.1f} kHz from the fitted f0 -- this is the "
      f"single-Lorentzian model error at the parked points, and it cancels out of every shift)")
print(f"\nCalibration JSON: {TWOPOINT_CALIB_JSON}")
print(f"Parked plan CSV:  {TWOPOINT_PLAN_CSV}")
display(FileLink(str(TWOPOINT_CALIB_JSON)))

# ------------------------------------------------------------------ plots ----
fig, (ax_all, ax_zoom) = plt.subplots(2, 1, figsize=(12, 8))

ax_all.plot(freqs_mhz, z_ref, color="0.3", lw=1.1, label="normalised PL (z scale)")
for _, r in df_scores.iterrows():
    _c = "tab:green" if r["eligible"] else "0.6"
    ax_all.axvline(r["center_mhz"], color=_c, ls=":", lw=1.0, alpha=0.8)
    ax_all.annotate(f"{r['total_score']:.2f}", xy=(r["center_mhz"], 1.0),
                    xycoords=("data", "axes fraction"),
                    xytext=(0, -4), textcoords="offset points", ha="center", va="top",
                    fontsize=7, color=_c, rotation=90)
ax_all.axvspan(f0 - CHOSEN_PEAK["window_mhz"] / 2, f0 + CHOSEN_PEAK["window_mhz"] / 2,
               color="tab:orange", alpha=0.12, label="tracked peak fit window")
ax_all.axvline(f_minus, color="tab:orange", ls="--", lw=1.4)
ax_all.axvline(f_plus,  color="tab:blue",   ls="--", lw=1.4)
ax_all.set(xlabel="MW frequency (MHz)", ylabel="z = orient x mw_on/|median(mw_off)|",
           title=f"All candidates (annotated with total_score) - tracking {f0:.3f} MHz")
ax_all.grid(True, alpha=0.3)
ax_all.legend(fontsize=8, loc="lower right")

_zoom_half = max(1.5 * CHOSEN_PEAK["fwhm_mhz"], 1.5 * (f_plus - f_minus))
_m = np.abs(freqs_mhz - f0) <= _zoom_half
_ffine = np.linspace(f0 - _zoom_half, f0 + _zoom_half, 800)
ax_zoom.plot(freqs_mhz[_m], z_ref[_m], "o-", ms=3, lw=1.0, color="0.3", label="measured")
ax_zoom.plot(_ffine, lorentzian_dip(_ffine, CHOSEN_PEAK["baseline"], CHOSEN_PEAK["contrast"],
                                    f0, gamma),
             color="crimson", lw=1.4,
             label=f"Lorentzian fit (FWHM {CHOSEN_PEAK['fwhm_mhz']:.3f} MHz)")
ax_zoom.axhline(CHOSEN_PEAK["baseline"], color="0.5", ls="--", lw=0.8, label="fitted baseline")
ax_zoom.axhline(CHOSEN_PEAK["baseline"] - 0.5 * CHOSEN_PEAK["contrast"], color="0.7", ls=":", lw=0.8,
                label="half depth")
ax_zoom.axvline(f0, color="purple", ls=":", lw=1.0, label=f"fitted f0 = {f0:.3f} MHz")
_c_mid = 0.5 * (f_minus + f_plus)
if abs(_c_mid - f0) > 1e-9:
    ax_zoom.axvline(_c_mid, color="darkgreen", ls="-.", lw=1.0,
                    label=f"balance centre = {_c_mid:.3f} MHz")
for _f, _b, _col, _lab in ((f_minus, baseline_minus, "tab:orange", "f-"),
                           (f_plus,  baseline_plus,  "tab:blue",   "f+")):
    ax_zoom.axvline(_f, color=_col, ls="--", lw=1.4)
    ax_zoom.plot([_f], [_b], "o", color=_col, ms=7, zorder=5)
    ax_zoom.annotate(f"{_lab} {_f:.3f}", xy=(_f, _b), xytext=(0, 12), textcoords="offset points",
                     ha="center", fontsize=8, color=_col)
ax_zoom.set(xlabel="MW frequency (MHz)", ylabel="z",
            title=f"Tracked peak and parked frequencies ({PLACEMENT_MODE})")
ax_zoom.grid(True, alpha=0.3)
ax_zoom.legend(fontsize=8, loc="lower right")

plt.tight_layout()
plt.show()

## Step 2 - One batch at the two parked frequencies

`MultipointLockinODMR` is generic in the number of frequencies, so the same single-program
upload used for 16 points works for 2. With only 2 points a batch is roughly 8x shorter
than the 16-point equivalent at the same `reps`, which is the whole reason for running a
single peak: the same averaging in an eighth of the time, or 8x the update rate at the
same noise.

This cell is the sanity check before going live - it confirms the parked ADC values land
on the ODMR flanks where Step 1 said they would.

In [ ]:
# Step 2 - Acquire the two parked frequencies in ONE FPGA program upload.
#
# DEFENSIVE GUARD (inherited from Lockin_module): the program is built ONCE before the
# batch loop. Each MultipointLockinODMR build is a ~300-500 ms FPGA upload, so rebuilding
# inside the loop is what made the old 8-peak cell take minutes.
import sys
from copy import copy
from pathlib import Path
from time import perf_counter, time
import numpy as np
import pandas as pd
from IPython.display import FileLink, display

# notebook_modules/ is already on sys.path via the project_root_setup cell.
import importlib
import multipoint_lockin_program
multipoint_lockin_program = importlib.reload(multipoint_lockin_program)
from multipoint_lockin_program import MultipointLockinODMR

if "TWOPOINT_FREQS_MHZ" not in globals():
    raise RuntimeError("Run Step 1 first -- it defines TWOPOINT_FREQS_MHZ and TWOPOINT_CALIB.")

# --- Acquisition parameters ---
TWOPOINT_REPS_PER_BATCH   = 10
TWOPOINT_N_BATCHES        = 1
TWOPOINT_OFF_RESONANCE_MHZ = 2700.0   # off-resonance reference frequency
TWOPOINT_RELAX_DELAY_TREG  = 1000     # delay between signal and off-resonance reference shots
TWOPOINT_DATA_CSV = TWOPOINT_DIR / "twopoint_lockin_collected.csv"
TWOPOINT_DATA_CSV.parent.mkdir(parents=True, exist_ok=True)

# --- Build the two-point config ---
cfg = copy(default_config)
cfg.multipoint_freqs_mhz = list(TWOPOINT_FREQS_MHZ)
cfg.odmr_reference_offres_mhz = TWOPOINT_OFF_RESONANCE_MHZ
cfg.relax_delay_treg = int(TWOPOINT_RELAX_DELAY_TREG)
# QickSweep needs a sweep definition even though we drive frequencies from body();
# a zero-span single point keeps the outer sweep trivial.
cfg.mw_start_fMHz = float(TWOPOINT_FREQS_MHZ[0])
cfg.mw_end_fMHz = float(TWOPOINT_FREQS_MHZ[0])
cfg.nsweep_points = 1
cfg.reps = int(TWOPOINT_REPS_PER_BATCH)
cfg.pre_init = getattr(default_config, "pre_init", True)

t_build_start = perf_counter()
prog = MultipointLockinODMR(cfg)
_build_seconds = perf_counter() - t_build_start
_predicted_ms_per_batch = prog.total_time() * 1e3
print(f"Built single MultipointLockinODMR program for {len(TWOPOINT_FREQS_MHZ)} frequencies "
      f"({TWOPOINT_FREQS_MHZ[0]:.3f}, {TWOPOINT_FREQS_MHZ[1]:.3f} MHz) in {_build_seconds:.2f} s")
print(f"Predicted time per batch: {_predicted_ms_per_batch:.0f} ms "
      f"({prog.time_per_rep()*1e3:.1f} ms/rep x {cfg.reps} reps)")

_first_batch_dt = None
rows = []
t0 = perf_counter()
for batch in range(int(TWOPOINT_N_BATCHES)):
    t_acq_start = perf_counter()
    d_batch = prog.acquire(progress=False)
    acq_dt = perf_counter() - t_acq_start
    if _first_batch_dt is None:
        _first_batch_dt = acq_dt

    row = {
        "batch": batch,
        "time_s": perf_counter() - t0,
        "timestamp_epoch_s": time(),
        "acq_seconds": acq_dt,
    }
    for k, (freq_mhz, sig, ref) in enumerate(zip(d_batch.frequencies_mhz, d_batch.signal, d_batch.reference)):
        row[f"peak_{k+1:02d}"] = float(sig)
        row[f"peak_{k+1:02d}_ref"] = float(ref)
        row[f"peak_{k+1:02d}_freq_mhz"] = float(freq_mhz)
    rows.append(row)
    print(f"batch {batch+1}/{int(TWOPOINT_N_BATCHES)}: acquired in {acq_dt*1000:.0f} ms "
          f"(predicted ~{_predicted_ms_per_batch:.0f} ms)")

df_twopoint = pd.DataFrame(rows)
df_twopoint.to_csv(TWOPOINT_DATA_CSV, index=False)
print(f"\nSaved {len(df_twopoint)} batches to {TWOPOINT_DATA_CSV}")

if _first_batch_dt * 1000 > _predicted_ms_per_batch * 5:
    print(f"WARNING: first batch took {_first_batch_dt*1000:.0f} ms vs predicted "
          f"{_predicted_ms_per_batch:.0f} ms. If this persists across runs, network/board "
          f"state is the likely cause.")

display(df_twopoint)
display(FileLink(str(TWOPOINT_DATA_CSV)))

# --- Overlay: do the parked ADC values land on the ODMR flanks? ---
_row = df_twopoint.iloc[0]
_pk_f = np.array([float(_row["peak_01_freq_mhz"]), float(_row["peak_02_freq_mhz"])])
_pk_s = np.array([float(_row["peak_01"]), float(_row["peak_02"])])
_pk_r = np.array([float(_row["peak_01_ref"]), float(_row["peak_02_ref"])])
_pk_z = TWOPOINT_CALIB["orient"] * _pk_s / np.abs(_pk_r)

_zoom_half = max(2.5 * TWOPOINT_CALIB["fwhm_mhz"], 1.5 * (TWOPOINT_CALIB["f_plus_mhz"] - TWOPOINT_CALIB["f_minus_mhz"]))
_m = np.abs(freqs_mhz - TWOPOINT_CALIB["f0_mhz"]) <= _zoom_half

fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(13, 4.2))
ax_l.plot(freqs_mhz, TWOPOINT_CALIB["orient"] * spectrum_norm, color="0.3", lw=1.0,
          label="reference ODMR (z)")
ax_l.scatter(_pk_f, _pk_z, color="tab:red", s=60, zorder=5, label="parked, this batch")
ax_l.set(xlabel="MW frequency (MHz)", ylabel="z", title="Full sweep")
ax_l.grid(True, alpha=0.3); ax_l.legend(fontsize=8)

ax_r.plot(freqs_mhz[_m], (TWOPOINT_CALIB["orient"] * spectrum_norm)[_m], "o-", ms=3, lw=1.0,
          color="0.3", label="reference ODMR (z)")
ax_r.scatter(_pk_f, _pk_z, color="tab:red", s=70, zorder=5, label="parked, this batch")
for _f, _b, _lab in ((TWOPOINT_CALIB["f_minus_mhz"], TWOPOINT_CALIB["baseline_minus"], "calib b-"),
                     (TWOPOINT_CALIB["f_plus_mhz"],  TWOPOINT_CALIB["baseline_plus"],  "calib b+")):
    ax_r.plot([_f], [_b], "x", color="tab:green", ms=10, mew=2, zorder=6, label=_lab)
ax_r.axvline(TWOPOINT_CALIB["f0_mhz"], color="purple", ls=":", lw=1.0)
ax_r.set(xlabel="MW frequency (MHz)", ylabel="z", title="Zoom on the tracked peak")
ax_r.grid(True, alpha=0.3); ax_r.legend(fontsize=8)
plt.tight_layout()
plt.show()

_dz = _pk_z - np.array([TWOPOINT_CALIB["baseline_minus"], TWOPOINT_CALIB["baseline_plus"]])
print(f"Parked z vs calibration baseline:  f- {_dz[0]:+.5f}   f+ {_dz[1]:+.5f}")
print("Both offsets should be small and of comparable size. A large COMMON offset means "
      "the laser/MW level drifted since the ODMR sweep (harmless for the exact method, "
      "which cancels contrast, but it biases the linear method). A large DIFFERENTIAL "
      "offset is a genuine peak shift.")

## Step 3 - Two-point conversion: parked PL -> peak frequency

Both conversions (defined at the end of Step 1, since they are pure functions of the
calibration) operate on the normalised, orientation-corrected signal
`z = orient x (signal / |reference|)`, the scale the Step 1 calibration was built on.

**Method A (exact).** With dip depths `d_i = B - z_i` and
`d_i = C g^2 / ((f_i - f0)^2 + g^2)`, the ratio `r = d_minus / d_plus` eliminates `C`:

`(r - 1) f0^2 - 2 (r f_minus - f_plus) f0 + (r f_minus^2 - f_plus^2 + (r - 1) g^2) = 0`

The physical root is the one nearest the parked midpoint. Only `g` and `B` enter, so
common-mode laser-power drift cancels. In the symmetric limit `r -> 1` the quadratic
degenerates and the linear root `-c/b` is used.

**Method B (linear).** `df = [(z_plus - z_minus) - (b_plus - b_minus)] / (m_minus - m_plus)`
using the measured flank baselines and slopes - the same expression as
`nv_toolkit.two_point.estimate_delta_f_mhz`, which is what `Lockin_module.ipynb` runs on
each of its 8 blocks.

**Shared zero.** Method B is zeroed on the measured baselines by construction. Method A is
therefore zeroed the same way: `f0_ref_exact_mhz`, the frequency the exact inversion
returns when fed the calibration baselines, is the reference the reported `delta_f` is
measured against - not the fitted `f0`. The gap between the two (printed in Step 1) is the
single-Lorentzian model error at the parked points; it is a constant and cancels out of
every shift, but leaving it in would offset A from B and destroy the cross-check.

A and B then agree while the shift is small compared to the linewidth. Persistent
disagreement means either a large shift (trust A) or a drifted baseline (trust neither
until you retake the ODMR sweep).

In [ ]:
# Step 3 - Apply the two-point conversion to the Step 2 batch.
# twopoint_z_from_counts / twopoint_f0_exact / twopoint_delta_f_linear are defined at the
# end of Step 1 (they are pure functions of TWOPOINT_CALIB); the maths is in the markdown
# above. Everything works in z = orient * signal / |reference|.
import numpy as np
import pandas as pd
from IPython.display import display

if "twopoint_f0_exact" not in globals():
    raise RuntimeError("Run Step 1 first -- it defines TWOPOINT_CALIB and the conversion helpers.")
if "df_twopoint" not in globals():
    raise RuntimeError("Run Step 2 first -- it defines df_twopoint.")


def twopoint_rows_from_wide(df_wide, calib=None):
    """Wide peak_01/peak_02 frame -> per-batch conversion table (both methods).

    delta_f is referenced to calib["f0_ref_exact_mhz"], the exact-inversion result at the
    calibration baselines, so methods A and B share a zero (see the markdown above).
    """
    calib = TWOPOINT_CALIB if calib is None else calib
    z_minus = twopoint_z_from_counts(df_wide["peak_01"].to_numpy(float),
                                     df_wide["peak_01_ref"].to_numpy(float), calib)
    z_plus = twopoint_z_from_counts(df_wide["peak_02"].to_numpy(float),
                                    df_wide["peak_02_ref"].to_numpy(float), calib)
    f0_exact = twopoint_f0_exact(z_minus, z_plus, calib)
    delta_f = f0_exact - float(calib["f0_ref_exact_mhz"])
    df_linear = twopoint_delta_f_linear(z_minus, z_plus, calib)
    gamma_nv = float(calib["gamma_nv_mhz_per_nT"])
    out = pd.DataFrame({
        "batch":                  df_wide["batch"].to_numpy() if "batch" in df_wide else np.arange(len(df_wide)),
        "time_s":                 df_wide["time_s"].to_numpy(float) if "time_s" in df_wide else np.arange(len(df_wide), dtype=float),
        "z_minus":                z_minus,
        "z_plus":                 z_plus,
        "lockin_signal":          z_plus - z_minus,
        "f0_mhz":                 f0_exact,
        "delta_f_mhz":            delta_f,
        "peak_shift_kHz":         delta_f * 1e3,
        "delta_f_linear_mhz":     df_linear,
        "peak_shift_linear_kHz":  df_linear * 1e3,
    })
    out["B_shift_nT"] = out["delta_f_mhz"] / gamma_nv
    out["B_shift_linear_nT"] = out["delta_f_linear_mhz"] / gamma_nv
    return out


df_convert = twopoint_rows_from_wide(df_twopoint)
TWOPOINT_CONVERT_CSV = TWOPOINT_DATA_CSV.with_name("twopoint_lockin_peak_inference.csv")
df_convert.to_csv(TWOPOINT_CONVERT_CSV, index=False)

print(f"Tracked peak: fitted f0 = {TWOPOINT_CALIB['f0_mhz']:.4f} MHz, "
      f"zero-point f0 = {TWOPOINT_CALIB['f0_ref_exact_mhz']:.4f} MHz, "
      f"FWHM = {TWOPOINT_CALIB['fwhm_mhz']:.4f} MHz")
display(df_convert.round(6))
print(f"\nWrote {TWOPOINT_CONVERT_CSV}")

_last = df_convert.iloc[-1]
_disagree_khz = abs(_last["peak_shift_kHz"] - _last["peak_shift_linear_kHz"])
_fwhm_khz = TWOPOINT_CALIB["fwhm_mhz"] * 1e3
print(f"\nLatest batch: f0 = {_last['f0_mhz']:.4f} MHz  "
      f"(shift {_last['peak_shift_kHz']:+.1f} kHz = {_last['B_shift_nT']:+.0f} nT along this NV axis, "
      f"{100 * abs(_last['peak_shift_kHz']) / _fwhm_khz:.1f}% of the linewidth)")
print(f"Method A vs B disagreement: {_disagree_khz:.2f} kHz "
      f"({'consistent' if _disagree_khz < 0.05 * _fwhm_khz else 'LARGE -- the shift is a big fraction of the linewidth, or the baseline drifted'})")
if not np.isfinite(_last["f0_mhz"]):
    print("f0 is NaN: at least one parked point sat at or above the fitted baseline, so the "
          "depth ratio has no physical root. Re-run Step 1 (the peak has moved off the "
          "parked pair) or check that the laser/MW level has not changed since the ODMR sweep.")

## Step 4 - Live continuous peak tracking

Keeps acquiring batch after batch and converts each one to a peak frequency on the fly,
streaming to CSV. This is the two-point equivalent of `lockin_live` in
`Lockin_module.ipynb`, and it carries over the same two fixes from there:

* **Peaking transient.** `pre_init=True` fires an MW polarisation pulse at every batch
  boundary, which perturbs the spins and produces a per-batch transient. The live program
  is built with `pre_init=False` and primed once with a throwaway `pre_init=True` acquire.
  An assertion enforces this.
* **Spike rejection.** A causal Hampel filter on the raw ADC (2 channels here instead of
  16) replaces single-batch glitches with the rolling per-channel median. Both the raw and
  the despiked values are recorded; `peak_NN_spike_*` flags mark every replacement.

`peak_shift_kHz` in the CSV is always referenced to `f0_ref_exact_mhz`, the Step 1
calibration zero shared by both methods - an absolute number that does not depend on when
the run started. Step 5 can re-reference to the run's first sample or its mean without
re-acquiring.

In [ ]:
# Step 4 - Live continuous two-point peak tracking.
#
# DEFENSIVE GUARDS (inherited from Lockin_module lockin_live):
#   - assert pre_init=False on the live program (peaking guard)
#   - the program is built BEFORE the loop, never inside it (runtime guard)
#   - watchdog that warns if a batch takes > LIVE_WATCHDOG_FACTOR x predicted time
#   - end-of-run summary with median/p95 batch times so drift is visible
import sys
from copy import copy
from pathlib import Path
from time import perf_counter, time as _wallclock
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import FileLink, display

if "twopoint_f0_exact" not in globals():
    raise RuntimeError("Run Step 1 once before live mode -- it defines TWOPOINT_CALIB and "
                       "the conversion helpers.")

import importlib
import multipoint_lockin_program
multipoint_lockin_program = importlib.reload(multipoint_lockin_program)
from multipoint_lockin_program import MultipointLockinODMR

import spike_rejection
spike_rejection = importlib.reload(spike_rejection)
from spike_rejection import HampelDespiker

# --- Live-mode parameters ---
LIVE_REPS_PER_BATCH     = 10     # reps inside each acquire() (smaller = faster, noisier)
LIVE_DURATION_SEC       = 30     # set None to run until the Interrupt button
LIVE_PLOT_REFRESH_EVERY = 1      # redraw every N batches (raise if plotting throttles the loop)
LIVE_SHOW_PLOT          = True   # False = headless; one static figure at the end
LIVE_OFF_RESONANCE_MHZ  = TWOPOINT_OFF_RESONANCE_MHZ if "TWOPOINT_OFF_RESONANCE_MHZ" in globals() else 2700.0
LIVE_RELAX_DELAY_TREG   = 1000   # laser repolarisation time between signal and reference
LIVE_WATCHDOG_FACTOR    = 3.0

# --- Spike rejection (causal Hampel filter on raw ADC, 2 channels) ---
LIVE_DESPIKE_ENABLED     = True
LIVE_DESPIKE_WINDOW      = 11    # trailing samples kept per channel
LIVE_DESPIKE_K_SIGMA     = 4     # rejection threshold in MAD-sigma units
LIVE_DESPIKE_WARMUP      = 4     # batches before rejection becomes active
LIVE_DESPIKE_SIGMA_FLOOR = 1.5   # minimum sigma estimate (ADC counts)
LIVE_DESPIKE_SIGMA_CAP   = 2     # maximum sigma estimate (ADC counts)

N_CHANNELS = 2

_run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
LIVE_CSV = TWOPOINT_DIR / f"twopoint_lockin_live_{_run_stamp}.csv"
LIVE_CSV.parent.mkdir(parents=True, exist_ok=True)

# --- Build the LIVE program with pre_init = False ---
cfg_live = copy(default_config)
cfg_live.multipoint_freqs_mhz      = list(TWOPOINT_FREQS_MHZ)
cfg_live.odmr_reference_offres_mhz = LIVE_OFF_RESONANCE_MHZ
cfg_live.relax_delay_treg          = int(LIVE_RELAX_DELAY_TREG)
cfg_live.mw_start_fMHz             = float(TWOPOINT_FREQS_MHZ[0])
cfg_live.mw_end_fMHz               = float(TWOPOINT_FREQS_MHZ[0])
cfg_live.nsweep_points             = 1
cfg_live.reps                      = int(LIVE_REPS_PER_BATCH)
cfg_live.pre_init                  = False

assert cfg_live.pre_init is False, (
    "Live mode requires pre_init=False to prevent per-batch peaking. An MW polarization "
    "pulse at start_freq before the body loop perturbs spins toward |+-1> at every batch "
    "boundary; with always-on laser the in-body readout windows then have to re-polarize, "
    "producing a transient."
)

prog_live = MultipointLockinODMR(cfg_live)
_predicted_ms_per_batch = prog_live.total_time() * 1e3
print(f"Built LIVE program (pre_init=False) for {len(TWOPOINT_FREQS_MHZ)} freqs "
      f"({TWOPOINT_FREQS_MHZ[0]:.3f}, {TWOPOINT_FREQS_MHZ[1]:.3f} MHz), "
      f"{cfg_live.reps} reps/batch, predicted ~{_predicted_ms_per_batch:.0f} ms/batch FPGA work")
print(f"Watchdog: will warn if any batch exceeds {_predicted_ms_per_batch * LIVE_WATCHDOG_FACTOR:.0f} ms.")

# --- One-shot priming acquire with pre_init=True (results discarded) ---
cfg_prime = copy(cfg_live)
cfg_prime.pre_init = True
prog_prime = MultipointLockinODMR(cfg_prime)
print("Priming spin polarization (one acquire with pre_init=True)...")
_ = prog_prime.acquire(progress=False)
del prog_prime, cfg_prime
print("Priming done. Entering live loop.\n")

# --- Despiker warm-up baseline from the Step 2 single batch, if available ---
_sig_baseline = None
_ref_baseline = None
_baseline_src = "(none)"
if LIVE_DESPIKE_ENABLED:
    try:
        _bl_row = pd.read_csv(TWOPOINT_DATA_CSV).iloc[-1]
        _sig_baseline = np.array([float(_bl_row[f"peak_{i:02d}"])     for i in range(1, N_CHANNELS + 1)])
        _ref_baseline = np.array([float(_bl_row[f"peak_{i:02d}_ref"]) for i in range(1, N_CHANNELS + 1)])
        _baseline_src = str(TWOPOINT_DATA_CSV)
    except Exception as _e:
        print(f"  (despike) no single-batch baseline available ({_e}); warm-up will pass samples through.")

_despike_kwargs = dict(
    n_channels=N_CHANNELS,
    window=LIVE_DESPIKE_WINDOW,
    k_sigma=LIVE_DESPIKE_K_SIGMA,
    min_warmup=LIVE_DESPIKE_WARMUP,
    sigma_floor=LIVE_DESPIKE_SIGMA_FLOOR,
    sigma_cap=LIVE_DESPIKE_SIGMA_CAP,
)
despiker_sig = HampelDespiker(baseline=_sig_baseline, **_despike_kwargs)
despiker_ref = HampelDespiker(baseline=_ref_baseline, **_despike_kwargs)
print(f"Spike rejection: enabled={LIVE_DESPIKE_ENABLED}, window={LIVE_DESPIKE_WINDOW}, "
      f"k_sigma={LIVE_DESPIKE_K_SIGMA}, warmup={LIVE_DESPIKE_WARMUP}, "
      f"sigma_floor={LIVE_DESPIKE_SIGMA_FLOOR}, baseline={_baseline_src}\n")

# --- Live plot: parked z / lock-in signal / peak shift ---
_gamma_nv = float(TWOPOINT_CALIB["gamma_nv_mhz_per_nT"])
if LIVE_SHOW_PLOT:
    plt.ion()
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    _l_zminus, = axes[0].plot([], [], "-", lw=1.0, color="tab:orange",
                              label=f"z at f- ({TWOPOINT_CALIB['f_minus_mhz']:.3f} MHz)")
    _l_zplus,  = axes[0].plot([], [], "-", lw=1.0, color="tab:blue",
                              label=f"z at f+ ({TWOPOINT_CALIB['f_plus_mhz']:.3f} MHz)")
    axes[0].axhline(TWOPOINT_CALIB["baseline_minus"], color="tab:orange", ls=":", lw=0.8)
    axes[0].axhline(TWOPOINT_CALIB["baseline_plus"],  color="tab:blue",   ls=":", lw=0.8)
    axes[0].set_ylabel("normalised PL (z)")
    axes[0].legend(fontsize=8, loc="upper right")

    _l_lockin, = axes[1].plot([], [], "-", lw=1.0, color="tab:green")
    axes[1].axhline(TWOPOINT_CALIB["baseline_plus"] - TWOPOINT_CALIB["baseline_minus"],
                    color="0.5", ls="--", lw=0.8, label="calibration reference")
    axes[1].set_ylabel("lock-in (z+ - z-)")
    axes[1].legend(fontsize=8, loc="upper right")

    _l_shift,  = axes[2].plot([], [], "-", lw=1.0, color="crimson", label="exact inversion (A)")
    _l_shiftl, = axes[2].plot([], [], "--", lw=0.9, color="0.5", alpha=0.8, label="linear estimate (B)")
    axes[2].axhline(0, color="0.4", lw=0.5)
    axes[2].set_ylabel("peak shift (kHz)")
    axes[2].set_xlabel("time (s)")
    axes[2].legend(fontsize=8, loc="upper right")
    _ax_b = axes[2].secondary_yaxis(
        "right", functions=(lambda k: k * 1e-3 / _gamma_nv, lambda n: n * _gamma_nv * 1e3))
    _ax_b.set_ylabel("equivalent dB along this NV axis (nT)")

    for _a in axes:
        _a.grid(True, alpha=0.3)
    fig.suptitle(f"Live two-point peak tracking, f0 = {TWOPOINT_CALIB['f0_mhz']:.4f} MHz "
                 f"-- {LIVE_CSV.name}")
    fig.tight_layout()
    _handle = display(fig, display_id=True)

# --- Live loop ---
rows = []
hist = {"t": [], "z_minus": [], "z_plus": [], "lockin": [], "shift": [], "shift_lin": []}
_acq_seconds_history = []
_n_watchdog_warnings = 0
_n_spike_sig_total = 0
_n_spike_ref_total = 0
_n_nan_f0 = 0
t0 = perf_counter()
batch = 0
last_save_t = t0
SAVE_EVERY_SEC = 5.0

try:
    while True:
        t_start = perf_counter() - t0
        d_batch = prog_live.acquire(progress=False)
        t_end = perf_counter() - t0
        t_mid = 0.5 * (t_start + t_end)
        acq_dt = t_end - t_start
        _acq_seconds_history.append(acq_dt)

        if acq_dt * 1000 > _predicted_ms_per_batch * LIVE_WATCHDOG_FACTOR and _n_watchdog_warnings < 5:
            print(f"  WARNING batch {batch}: acq_seconds={acq_dt*1000:.0f} ms "
                  f"(predicted ~{_predicted_ms_per_batch:.0f} ms). Likely network jitter, not a code bug.")
            _n_watchdog_warnings += 1
            if _n_watchdog_warnings == 5:
                print("  (further watchdog warnings suppressed; full timings in the summary at the end)")

        # ---- Spike rejection on raw ADC ----
        sig_raw = np.asarray(d_batch.signal, dtype=float)
        ref_raw = np.asarray(d_batch.reference, dtype=float)
        if LIVE_DESPIKE_ENABLED:
            sig_clean, sig_flags = despiker_sig.update(sig_raw)
            ref_clean, ref_flags = despiker_ref.update(ref_raw)
        else:
            sig_clean, ref_clean = sig_raw, ref_raw
            sig_flags = np.zeros(N_CHANNELS, dtype=bool)
            ref_flags = np.zeros(N_CHANNELS, dtype=bool)
        _n_spike_sig_total += int(sig_flags.sum())
        _n_spike_ref_total += int(ref_flags.sum())

        row = {
            "batch": batch,
            "time_s": t_mid,
            "timestamp_epoch_s": _wallclock(),
            "acq_seconds": acq_dt,
        }
        # Wide-format peak columns hold the DESPIKED counts (so the conversion below uses
        # the cleaned signal); the raw values are kept alongside for auditing.
        for k_idx, (f, s, r, s_raw, r_raw) in enumerate(
            zip(d_batch.frequencies_mhz, sig_clean, ref_clean, sig_raw, ref_raw)
        ):
            row[f"peak_{k_idx+1:02d}"]           = float(s)
            row[f"peak_{k_idx+1:02d}_ref"]       = float(r)
            row[f"peak_{k_idx+1:02d}_freq_mhz"]  = float(f)
            row[f"peak_{k_idx+1:02d}_raw"]       = float(s_raw)
            row[f"peak_{k_idx+1:02d}_ref_raw"]   = float(r_raw)
            row[f"peak_{k_idx+1:02d}_spike_sig"] = bool(sig_flags[k_idx])
            row[f"peak_{k_idx+1:02d}_spike_ref"] = bool(ref_flags[k_idx])

        # ---- On-the-fly conversion (both methods) ----
        z_minus = float(twopoint_z_from_counts(row["peak_01"], row["peak_01_ref"]))
        z_plus  = float(twopoint_z_from_counts(row["peak_02"], row["peak_02_ref"]))
        f0_exact = float(twopoint_f0_exact(z_minus, z_plus))
        df_lin = float(twopoint_delta_f_linear(z_minus, z_plus))
        delta_f = f0_exact - float(TWOPOINT_CALIB["f0_ref_exact_mhz"])
        if not np.isfinite(f0_exact):
            _n_nan_f0 += 1

        row.update({
            "z_minus":               z_minus,
            "z_plus":                z_plus,
            "lockin_signal":         z_plus - z_minus,
            "f0_mhz":                f0_exact,
            "delta_f_mhz":           delta_f,
            "peak_shift_kHz":        delta_f * 1e3,
            "delta_f_linear_mhz":    df_lin,
            "peak_shift_linear_kHz": df_lin * 1e3,
            "B_shift_nT":            delta_f / _gamma_nv,
            "B_shift_linear_nT":     df_lin / _gamma_nv,
        })
        rows.append(row)

        hist["t"].append(t_mid)
        hist["z_minus"].append(z_minus)
        hist["z_plus"].append(z_plus)
        hist["lockin"].append(z_plus - z_minus)
        hist["shift"].append(delta_f * 1e3)
        hist["shift_lin"].append(df_lin * 1e3)

        if LIVE_SHOW_PLOT and batch % LIVE_PLOT_REFRESH_EVERY == 0:
            _l_zminus.set_data(hist["t"], hist["z_minus"])
            _l_zplus.set_data(hist["t"], hist["z_plus"])
            _l_lockin.set_data(hist["t"], hist["lockin"])
            _l_shift.set_data(hist["t"], hist["shift"])
            _l_shiftl.set_data(hist["t"], hist["shift_lin"])
            for _a in axes:
                _a.relim()
                _a.autoscale_view()
            fig.canvas.draw_idle()
            _handle.update(fig)

        if perf_counter() - last_save_t > SAVE_EVERY_SEC:
            pd.DataFrame(rows).to_csv(LIVE_CSV, index=False)
            last_save_t = perf_counter()

        batch += 1
        if LIVE_DURATION_SEC is not None and t_end >= LIVE_DURATION_SEC:
            break
except KeyboardInterrupt:
    print(f"Live stopped by user -- {batch} batches collected.")
finally:
    if LIVE_SHOW_PLOT:
        plt.ioff()

# --- Final save + summary ---
df_live = pd.DataFrame(rows)
df_live.to_csv(LIVE_CSV, index=False)

print(f"\nLive run finished: {len(df_live)} batches in {df_live['time_s'].iloc[-1]:.1f} s "
      f"({len(df_live)/df_live['time_s'].iloc[-1]:.1f} Hz update rate)")
print(f"Saved to: {LIVE_CSV}")
display(FileLink(str(LIVE_CSV)))

_acq_arr = np.asarray(_acq_seconds_history) * 1000.0
print(f"\nPer-batch timing (ms): predicted ~{_predicted_ms_per_batch:.0f}  "
      f"actual median={np.median(_acq_arr):.0f}, p95={np.percentile(_acq_arr, 95):.0f}, "
      f"max={_acq_arr.max():.0f}")
if _n_watchdog_warnings:
    print(f"Watchdog fired {_n_watchdog_warnings} time(s) out of {len(_acq_arr)} batches "
          f"(network jitter; the FPGA pulse work is constant).")
else:
    print("Watchdog never fired -- runtime is consistent with prediction.")

if LIVE_DESPIKE_ENABLED and len(df_live):
    _cells = len(df_live) * N_CHANNELS
    print(f"\nSpike rejection: {_n_spike_sig_total} signal + {_n_spike_ref_total} reference "
          f"rejections ({100*_n_spike_sig_total/_cells:.2f}% / {100*_n_spike_ref_total/_cells:.2f}% of cells).")

_sh = df_live["peak_shift_kHz"].to_numpy(float)
_sh_ok = _sh[np.isfinite(_sh)]
if _n_nan_f0:
    print(f"\n{_n_nan_f0}/{len(df_live)} batches gave NaN f0 (a parked point sat above the "
          f"fitted baseline). Those batches are excluded from the statistics below.")
if _sh_ok.size:
    print(f"\nPeak shift vs calibration f0: mean {_sh_ok.mean():+.1f} kHz, "
          f"std {_sh_ok.std():.1f} kHz, pk-pk {np.ptp(_sh_ok):.1f} kHz")
    print(f"Equivalent field along this NV axis: std {_sh_ok.std()*1e-3/_gamma_nv:.0f} nT")

if not LIVE_SHOW_PLOT and len(df_live):
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    axes[0].plot(df_live["time_s"], df_live["z_minus"], lw=0.9, color="tab:orange", label="z at f-")
    axes[0].plot(df_live["time_s"], df_live["z_plus"],  lw=0.9, color="tab:blue",   label="z at f+")
    axes[0].set_ylabel("normalised PL (z)"); axes[0].legend(fontsize=8)
    axes[1].plot(df_live["time_s"], df_live["lockin_signal"], lw=0.9, color="tab:green")
    axes[1].set_ylabel("lock-in (z+ - z-)")
    axes[2].plot(df_live["time_s"], df_live["peak_shift_kHz"], lw=0.9, color="crimson",
                 label="exact inversion (A)")
    axes[2].plot(df_live["time_s"], df_live["peak_shift_linear_kHz"], "--", lw=0.8, color="0.5",
                 label="linear estimate (B)")
    axes[2].axhline(0, color="0.4", lw=0.5)
    axes[2].set_ylabel("peak shift (kHz)"); axes[2].set_xlabel("time (s)"); axes[2].legend(fontsize=8)
    _ax_b = axes[2].secondary_yaxis(
        "right", functions=(lambda k: k * 1e-3 / _gamma_nv, lambda n: n * _gamma_nv * 1e3))
    _ax_b.set_ylabel("equivalent dB (nT)")
    for _a in axes:
        _a.grid(True, alpha=0.3)
    fig.suptitle(f"Two-point peak tracking (headless) -- {LIVE_CSV.name}")
    plt.tight_layout()
    plt.show()

## Step 5 - Post-processing the live run

Reads the CSV written by Step 4 (or any earlier one via `LIVE_CSV_OVERRIDE`) and produces
the four-panel summary, the equivalent of the QDM-heart peak-shift figure plus a noise
spectrum:

1. normalised PL at both parked frequencies, against the calibration baselines
2. lock-in signal `z+ - z-`
3. peak shift in kHz, with the equivalent NV-axis field on the right axis
4. amplitude spectral density of the shift, in nT/sqrt(Hz)

`SHIFT_REFERENCE` re-zeroes the trace: `"calibration"` keeps the absolute shift against the
Step 1 ODMR fit, `"first"` and `"mean"` remove the constant offset so only the time
structure remains. The offset between `"calibration"` and `"mean"` is a real number worth
reading - it is the drift between the ODMR sweep and the run.

In [ ]:
# Step 5 - Post-processing: peak-shift time series, statistics and noise spectrum.
# Works standalone on any CSV written by Step 4 (all derived columns are already in it).
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import FileLink, display

LIVE_CSV_OVERRIDE = None      # Path("...twopoint_lockin_live_....csv") to process an older run
SHIFT_REFERENCE = "calibration"   # "calibration" | "first" | "mean"
ASD_DETREND = True                # remove a linear drift before the spectrum

# --- Resolve which CSV to process ---
if LIVE_CSV_OVERRIDE is not None:
    _csv_to_load = Path(LIVE_CSV_OVERRIDE)
elif "LIVE_CSV" in globals() and Path(LIVE_CSV).exists():
    _csv_to_load = Path(LIVE_CSV)
else:
    _dir = PROJECT_ROOT / "data" / "twopoint_lockin"
    _candidates = sorted(_dir.glob("twopoint_lockin_live_*.csv"))
    if not _candidates:
        raise FileNotFoundError(f"No twopoint_lockin_live_*.csv found in {_dir}")
    _csv_to_load = _candidates[-1]
print(f"Processing: {_csv_to_load.name}")

dfl = pd.read_csv(_csv_to_load)
if "peak_shift_kHz" not in dfl.columns:
    raise RuntimeError("No peak_shift_kHz column -- was this CSV produced by Step 4?")

t = dfl["time_s"].to_numpy(float)
ok = np.isfinite(dfl["peak_shift_kHz"].to_numpy(float))
n_bad = int((~ok).sum())
print(f"  {len(dfl)} batches over {t[-1]:.1f} s "
      f"({len(dfl)/t[-1]:.1f} Hz)" + (f", {n_bad} with undefined f0 (excluded)" if n_bad else ""))

# --- Re-reference the shift ---
_shift_a = dfl["peak_shift_kHz"].to_numpy(float)
_shift_b = dfl["peak_shift_linear_kHz"].to_numpy(float)
if SHIFT_REFERENCE == "calibration":
    _zero = 0.0
elif SHIFT_REFERENCE == "first":
    _zero = float(_shift_a[ok][0])
elif SHIFT_REFERENCE == "mean":
    _zero = float(np.nanmean(_shift_a[ok]))
else:
    raise ValueError(f"Unknown SHIFT_REFERENCE: {SHIFT_REFERENCE!r}")
shift_a = _shift_a - _zero
shift_b = _shift_b - _zero

# gamma_NV: prefer the run's own calibration, else the module default.
_gamma_nv = float(TWOPOINT_CALIB["gamma_nv_mhz_per_nT"]) if "TWOPOINT_CALIB" in globals() else 28.024e-6
B_nT = shift_a * 1e-3 / _gamma_nv

# --- Statistics ---
_a, _b = shift_a[ok], shift_b[ok]
_dt = float(np.median(np.diff(t))) if len(t) > 1 else float("nan")
_rate = 1.0 / _dt if _dt > 0 else float("nan")
_sigma_khz = float(_a.std())
_sigma_nT = _sigma_khz * 1e-3 / _gamma_nv
# White-noise sensitivity: sigma over one sample of duration dt corresponds to
# sigma * sqrt(2 * dt) in nT/sqrt(Hz) (single-sided, sqrt(2 dt) = 1/sqrt(bandwidth)).
_sens_nT_rtHz = _sigma_nT * np.sqrt(2.0 * _dt) if np.isfinite(_dt) else float("nan")

print(f"\nReference: {SHIFT_REFERENCE}" + ("" if SHIFT_REFERENCE == "calibration" else
      f" (removed a constant {_zero:+.1f} kHz vs the Step 1 calibration zero)"))
print(f"Peak shift: mean {_a.mean():+.2f} kHz, std {_sigma_khz:.2f} kHz, "
      f"pk-pk {np.ptp(_a):.2f} kHz")
print(f"Equivalent field along this NV axis: std {_sigma_nT:.0f} nT, "
      f"pk-pk {np.ptp(_a)*1e-3/_gamma_nv:.0f} nT")
print(f"Sample rate {_rate:.1f} Hz -> white-noise floor ~{_sens_nT_rtHz*1e3:.0f} pT/sqrt(Hz) "
      f"(valid only if the trace is white; read panel 4)")
_disagree = np.abs(_a - _b)
print(f"Method A vs B disagreement: median {np.median(_disagree):.2f} kHz, max {_disagree.max():.2f} kHz")
if "TWOPOINT_CALIB" in globals():
    _fwhm_khz = TWOPOINT_CALIB["fwhm_mhz"] * 1e3
    print(f"Shift range is {100*np.ptp(_a)/_fwhm_khz:.1f}% of the {_fwhm_khz:.0f} kHz linewidth "
          f"({'linear regime, A and B should agree' if np.ptp(_a) < 0.2*_fwhm_khz else 'outside the linear regime -- trust method A'})")

# --- Amplitude spectral density of the shift ---
_asd_f = _asd = None
if ok.sum() >= 16 and np.isfinite(_dt) and _dt > 0:
    x = B_nT[ok].astype(float)
    if ASD_DETREND:
        _p = np.polyfit(t[ok], x, 1)
        x = x - np.polyval(_p, t[ok])
    n = x.size
    w = np.hanning(n)
    X = np.fft.rfft(x * w)
    _asd_f = np.fft.rfftfreq(n, _dt)
    psd = 2.0 * np.abs(X) ** 2 * _dt / (n * np.mean(w ** 2))   # one-sided, nT^2/Hz
    _asd = np.sqrt(psd)
    _asd_f, _asd = _asd_f[1:], _asd[1:]                        # drop DC

# --- Four-panel figure ---
fig, axes = plt.subplots(2, 2, figsize=(15, 8))
ax0, ax1, ax2, ax3 = axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]

_f_minus = float(dfl["peak_01_freq_mhz"].iloc[0]) if "peak_01_freq_mhz" in dfl else float("nan")
_f_plus = float(dfl["peak_02_freq_mhz"].iloc[0]) if "peak_02_freq_mhz" in dfl else float("nan")
ax0.plot(t, dfl["z_minus"], lw=0.9, color="tab:orange", label=f"f- ({_f_minus:.3f} MHz)")
ax0.plot(t, dfl["z_plus"],  lw=0.9, color="tab:blue",   label=f"f+ ({_f_plus:.3f} MHz)")
if "TWOPOINT_CALIB" in globals():
    ax0.axhline(TWOPOINT_CALIB["baseline_minus"], color="tab:orange", ls=":", lw=0.9)
    ax0.axhline(TWOPOINT_CALIB["baseline_plus"],  color="tab:blue",   ls=":", lw=0.9)
ax0.set(title="Normalised PL at the parked frequencies", xlabel="time (s)", ylabel="z")
ax0.legend(fontsize=8)

ax1.plot(t, dfl["lockin_signal"], lw=0.9, color="tab:green")
if "TWOPOINT_CALIB" in globals():
    ax1.axhline(TWOPOINT_CALIB["baseline_plus"] - TWOPOINT_CALIB["baseline_minus"],
                color="0.5", ls="--", lw=0.9, label="calibration reference")
    ax1.legend(fontsize=8)
ax1.set(title="Lock-in signal (z+ - z-)", xlabel="time (s)", ylabel="differential z")

ax2.plot(t, shift_a, lw=0.9, color="crimson", label="exact inversion (A)")
ax2.plot(t, shift_b, "--", lw=0.8, color="0.5", alpha=0.8, label="linear estimate (B)")
ax2.axhline(0, color="0.4", lw=0.6)
ax2.set(title=f"Peak frequency shift (ref: {SHIFT_REFERENCE})", xlabel="time (s)",
        ylabel="peak shift (kHz)")
_axb = ax2.secondary_yaxis("right",
                           functions=(lambda k: k * 1e-3 / _gamma_nv,
                                      lambda n: n * _gamma_nv * 1e3))
_axb.set_ylabel("equivalent dB along this NV axis (nT)")
ax2.legend(fontsize=8)

if _asd is not None:
    ax3.loglog(_asd_f, _asd, lw=0.9, color="tab:purple")
    ax3.axhline(_sens_nT_rtHz, color="0.4", ls="--", lw=0.9,
                label=f"white-noise level {_sens_nT_rtHz*1e3:.0f} pT/rtHz")
    ax3.set(title="Amplitude spectral density of the peak shift",
            xlabel="frequency (Hz)", ylabel="nT / sqrt(Hz)")
    ax3.legend(fontsize=8)
else:
    ax3.text(0.5, 0.5, "not enough samples for an ASD", ha="center", va="center",
             transform=ax3.transAxes)
    ax3.set_axis_off()

for _a_ in (ax0, ax1, ax2, ax3):
    _a_.grid(True, alpha=0.3, which="both")
fig.suptitle(f"Two-point peak tracking -- {_csv_to_load.name}", y=1.00)
plt.tight_layout()
plt.show()

# --- Save the processed table + a one-row summary ---
out = pd.DataFrame({
    "time_s": t,
    "z_minus": dfl["z_minus"],
    "z_plus": dfl["z_plus"],
    "lockin_signal": dfl["lockin_signal"],
    "f0_mhz": dfl["f0_mhz"],
    "peak_shift_kHz": shift_a,
    "peak_shift_linear_kHz": shift_b,
    "B_shift_nT": B_nT,
})
PROCESSED_CSV = _csv_to_load.with_name(_csv_to_load.stem + f"_peakshift_{SHIFT_REFERENCE}.csv")
out.to_csv(PROCESSED_CSV, index=False)

SUMMARY_CSV = _csv_to_load.with_name(_csv_to_load.stem + "_summary.csv")
pd.DataFrame([{
    "csv": _csv_to_load.name,
    "n_batches": len(dfl),
    "n_bad_f0": n_bad,
    "duration_s": float(t[-1]),
    "rate_Hz": _rate,
    "shift_reference": SHIFT_REFERENCE,
    "shift_mean_kHz": float(_a.mean()),
    "shift_std_kHz": _sigma_khz,
    "shift_ptp_kHz": float(np.ptp(_a)),
    "B_std_nT": _sigma_nT,
    "sensitivity_nT_per_rtHz": _sens_nT_rtHz,
    "method_AB_median_disagreement_kHz": float(np.median(_disagree)),
    "f0_calibration_mhz": float(TWOPOINT_CALIB["f0_mhz"]) if "TWOPOINT_CALIB" in globals() else np.nan,
    "fwhm_calibration_mhz": float(TWOPOINT_CALIB["fwhm_mhz"]) if "TWOPOINT_CALIB" in globals() else np.nan,
}]).to_csv(SUMMARY_CSV, index=False)

print(f"\nSaved {PROCESSED_CSV}")
print(f"Saved {SUMMARY_CSV}")
display(FileLink(str(PROCESSED_CSV)))
display(FileLink(str(SUMMARY_CSV)))

## Caveats

- **No vector field.** Two parked points constrain one resonance frequency. The `B_shift_nT`
  column is `df / gamma_NV` - the field change projected on the tracked NV axis, and only
  under the assumption that the shift is Zeeman in origin. Temperature drift moves `D` and
  therefore moves every resonance too, and a single peak cannot tell the two apart. Use
  `Lockin_module.ipynb` when you need `dBx, dBy, dBz`.
- **The calibration is the ODMR sweep.** FWHM, baseline and flank slopes all come from one
  spectrum. Retake it whenever the bias field, MW power or laser power changes. The Step 2
  overlay is the check: a large common offset between the parked z values and the
  calibration baselines means the reference has gone stale.
- **Method A cancels contrast, not linewidth.** Common-mode laser-power drift drops out of
  the depth ratio, but a wrong FWHM rescales every shift by a constant factor. The shape of
  the trace is trustworthy before the kHz axis is.
- **Single-Lorentzian model.** Unresolved 14N hyperfine (three lines, 2.16 MHz apart) makes a
  narrow peak non-Lorentzian in detail, which biases the absolute `f0` more than the relative
  shift. If the sweep resolves the structure, narrow `FIT_WINDOW_MHZ` onto one component and
  park inside it - the symmetry score in Step 1 is there to steer you toward peaks where
  this is least bad.
- **Linear range.** Method B is only valid while the shift is a small fraction of the
  linewidth. Step 5 prints the shift range as a percentage of the FWHM; past ~20% believe
  method A and treat B as a diagnostic.
- **Out-of-range shifts give NaN.** If the peak moves far enough that a parked point rises
  above the fitted baseline, the depth ratio has no physical root and `f0` is NaN. Widen the
  parked spacing (`PLACEMENT_MODE = "symmetric_half_max"` parks further out) or re-run
  Step 1 at the new field.